In [20]:
import glob
import pandas as pd
from functools import reduce
import os
import geopandas as gpd

pasta_arabica = 'dados-brutos/arabica'
pasta_canephora = 'dados-brutos/canephora'

arquivos_arabica = glob.glob(os.path.join(pasta_arabica, '*.xlsx'))
arquivos_canephora = glob.glob(os.path.join(pasta_canephora, '*.xlsx'))

In [21]:
def trata_excel(caminho):
    
    df = pd.read_excel(caminho, header=None)
    
    linha1 = df.iloc[1]  # variável
    linha3 = df.iloc[3]  # ano
    linha4 = df.iloc[4]  # tipo
    
    linha1 = linha1.ffill()
    linha3 = linha3.ffill()
    linha4 = linha4.ffill()
    
    novas_colunas = [
        f"{nome}_{ano}_{tipo}" if pd.notna(ano) else nome
        for nome, ano, tipo in zip(linha1, linha3, linha4)
    ]
    
    df.columns = novas_colunas
    
    df = df.iloc[5:].reset_index(drop=True)
    
    df.rename(columns={df.columns[0]: 'mesorregiao'}, inplace=True)
    
    df = df.dropna(how='all')
    
    df_long = df.melt(
        id_vars=['mesorregiao'],
        var_name='variavel_ano_tipo',
        value_name='valor'
    )
    
    df_long[['variavel', 'ano', 'tipo']] = df_long['variavel_ano_tipo'].str.rsplit('_', n=2, expand=True)
    
    df_long = df_long.drop(columns=['variavel_ano_tipo'])
    df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')


    df_final = df_long.pivot_table(
        index=['mesorregiao', 'ano', 'tipo'],
        columns='variavel',
        values='valor'
    ).reset_index()
    
    return df_final

In [24]:
dfs_arabica = []

for arquivo in arquivos_arabica:
    
    try:
        df = trata_excel(arquivo)
        
        dfs_arabica.append(df)
        
    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

dfs_canephora = []

for arquivo in arquivos_canephora:
    
    try:
        df = trata_excel(arquivo)
        
        dfs_canephora.append(df)
        
    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

In [25]:
dfs_arabica[0]

variavel,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare)
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0
...,...,...,...,...
819,Zona da Mata (MG),2020,Café (em grão) Arábica,1786.0
820,Zona da Mata (MG),2021,Café (em grão) Arábica,1087.0
821,Zona da Mata (MG),2022,Café (em grão) Arábica,1338.0
822,Zona da Mata (MG),2023,Café (em grão) Arábica,1299.0


In [26]:
dfs_canephora[0]

variavel,mesorregiao,ano,tipo,Variável - Área destinada à colheita (Hectares)
0,Baixo Amazonas (PA),2012,Café (em grão) Canephora,255.0
1,Baixo Amazonas (PA),2013,Café (em grão) Canephora,213.0
2,Baixo Amazonas (PA),2014,Café (em grão) Canephora,138.0
3,Baixo Amazonas (PA),2015,Café (em grão) Canephora,56.0
4,Baixo Amazonas (PA),2016,Café (em grão) Canephora,60.0
...,...,...,...,...
402,Zona da Mata (MG),2020,Café (em grão) Canephora,522.0
403,Zona da Mata (MG),2021,Café (em grão) Canephora,504.0
404,Zona da Mata (MG),2022,Café (em grão) Canephora,488.0
405,Zona da Mata (MG),2023,Café (em grão) Canephora,498.0


In [27]:
df_final_arabica = reduce(lambda left, right: left.merge(right, on=['mesorregiao', 'ano', 'tipo']), dfs_arabica)
df_final_canephora = reduce(lambda left, right: left.merge(right, on=['mesorregiao', 'ano', 'tipo']), dfs_canephora)

In [28]:
df_final = pd.concat([df_final_arabica, df_final_canephora], ignore_index=True)

In [29]:
df_final['Variável - Valor da produção (Mil Reais)'] = df_final['Variável - Valor da produção (Mil Reais)'] * 1000
df_final['Variável - Quantidade produzida (Toneladas)'] = df_final['Variável - Quantidade produzida (Toneladas)'] * 1000

In [30]:
df_final.rename(columns={'Variável - Valor da produção (Mil Reais)': 'Variável - Valor da produção (Reais)',
                         'Variável - Quantidade produzida (Toneladas)': 'Variável - Quantidade produzida (Quilogramas)'
                         }, inplace=True)

In [31]:
df_final

variavel,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas)
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0,36000.0,2.0,2.0,2000.0
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0,3851000.0,2733.0,2733.0,1075000.0
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0,2439000.0,2615.0,2615.0,771000.0
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1618000.0,2549.0,2549.0,693000.0
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0,2325000.0,2265.0,2265.0,867000.0
...,...,...,...,...,...,...,...,...
1224,Zona da Mata (MG),2020,Café (em grão) Canephora,2362.0,6623000.0,522.0,522.0,1233000.0
1225,Zona da Mata (MG),2021,Café (em grão) Canephora,2121.0,8679000.0,504.0,504.0,1069000.0
1226,Zona da Mata (MG),2022,Café (em grão) Canephora,2746.0,15327000.0,488.0,488.0,1340000.0
1227,Zona da Mata (MG),2023,Café (em grão) Canephora,2371.0,10643000.0,498.0,498.0,1181000.0


In [33]:
caminho_shapefile = "dados-brutos/centroides/BR_Mesorregioes_2022/BR_Mesorregioes_2022.shp"
mesorregioes = gpd.read_file(caminho_shapefile)
mesorregioes['centroide'] = mesorregioes.geometry.centroid
mesorregioes['latitude'] = mesorregioes.centroide.y
mesorregioes['longitude'] = mesorregioes.centroide.x

mesorregioes['NM_MESO'] = mesorregioes['NM_MESO'] + ' (' + mesorregioes['SIGLA_UF'] + ')'

mesorregioes = mesorregioes[['NM_MESO', 'latitude', 'longitude']]

/var/folders/wp/hrfyx8x11jlbkyy3vr606yph0000gn/T/ipykernel_5045/726320708.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  mesorregioes['centroide'] = mesorregioes.geometry.centroid


In [34]:
mesorregioes.rename(columns={'NM_MESO': 'mesorregiao'}, inplace=True)
mesorregioes.head()

,mesorregiao,latitude,longitude
0,Madeira-Guaporé (RO),-10.302659,-64.055361
1,Leste Rondoniense (RO),-11.406309,-61.861723
2,Vale do Juruá (AC),-8.539777,-71.773867
3,Vale do Acre (AC),-9.941532,-69.066201
4,Norte Amazonense (AM),-0.615226,-65.567522


In [35]:
df_final = df_final.merge(mesorregioes, on='mesorregiao', how='left')

In [36]:
df_final.head()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0,36000.0,2.0,2.0,2000.0,-7.029713,-35.819545
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0,3851000.0,2733.0,2733.0,1075000.0,-8.510505,-36.416511
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0,2439000.0,2615.0,2615.0,771000.0,-8.510505,-36.416511
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1618000.0,2549.0,2549.0,693000.0,-8.510505,-36.416511
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0,2325000.0,2265.0,2265.0,867000.0,-8.510505,-36.416511


In [37]:
def trata_temperatura(caminho):

    cabecalho = pd.read_csv(
        caminho,
        sep=';',
        encoding='latin1',
        nrows=8,
        header=None
    )


    latitude = cabecalho.iloc[4, 1]
    longitude = cabecalho.iloc[5, 1]
    altitude  = cabecalho.iloc[6, 1]


    df = pd.read_csv(
    caminho,
    sep=';',
    encoding='latin1',
    skiprows=8  
    )

    df['latitude'] = latitude
    df['longitude'] = longitude
    df['altitude'] = altitude

    nome_arquivo = os.path.basename(caminho)

    partes = nome_arquivo.split('_')

    regiao = partes[1]
    uf = partes[2]
    cidade = partes[4]
    ano = partes[5]

    df['regiao'] = regiao
    df['uf'] = uf
    df['cidade'] = cidade
    df['ano'] = ano

    if ano[-4:] in ['2019', '2020', '2021', '2022', '2023', '2024']:
        df['DATA (YYYY-MM-DD)'] = pd.to_datetime(df['Data']).dt.date

    df = df[['DATA (YYYY-MM-DD)', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)', 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
          'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)', 'UMIDADE RELATIVA DO AR, HORARIA (%)', 
          'regiao', 'uf', 'cidade', 'ano', 'latitude', 'longitude', 'altitude']]
    
    cols = [
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'altitude'
    ]

    for col in cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.'),
            errors='coerce'
        )

    df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].astype(float)
    df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] = df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'].astype(float)
    df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] = df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'].astype(float)
    df['altitude'] = df['altitude'].astype(float)

    df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].clip(lower=0)
    df = df[df['TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'] >= -100]
    df = df[df['TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'] >= -100]
    df = df[df['UMIDADE RELATIVA DO AR, HORARIA (%)'] >= 0]
    df = df[df['UMIDADE RELATIVA DO AR, HORARIA (%)'] <= 100]
    df = df[df['altitude'] >= 0]

    df_diario = df.groupby('DATA (YYYY-MM-DD)').agg({
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'sum',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'max',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'min',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'mean',
        'altitude': 'mean'
    }).reset_index()


    df_diario['DATA (YYYY-MM-DD)'] = pd.to_datetime(df_diario['DATA (YYYY-MM-DD)'])
    mask = df_diario['DATA (YYYY-MM-DD)'].dt.month >= 10
    soma = df_diario.loc[mask, 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].sum()
    df_diario['precipitacao_floracao'] = soma

    df_final = df_diario.agg({
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'sum',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'mean',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'mean',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'mean',
        'precipitacao_floracao': 'mean',
        'altitude': 'mean'
    }).to_frame().T


    df_final['regiao'] = df['regiao'].iloc[0]
    df_final['uf'] = df['uf'].iloc[0]
    df_final['cidade'] = df['cidade'].iloc[0]
    df_final['ano'] = df['ano'].iloc[0]
    df_final['latitude'] = df['latitude'].iloc[0]
    df_final['longitude'] = df['longitude'].iloc[0]

    df_final = df_final.rename(columns={
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao_total (mm)',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temperatura_maxima (°C)',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temperatura_minima (°C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_relativa (%)'})

    df_final['ano'] = pd.to_datetime(df_final['ano'], format='%d-%m-%Y')
    df_final['ano'] = df_final['ano'].dt.year

    return df_final

In [38]:
dfs = []

for root, dirs, files in os.walk('dados-brutos/dados-climaticos'):
    for file in files:
        if file.endswith('.CSV'):
            
            caminho = os.path.join(root, file)
            
            try:
                df = trata_temperatura(caminho)
                
                if df is not None:
                    dfs.append(df)
            
            except Exception as e:
                print(f"Erro em {caminho}: {e}")

df_temp = pd.concat(dfs, ignore_index=True)

Erro em dados-brutos/dados-climaticos/2013/INMET_SE_MG_F501_BELO HORIZONTE - CERCADINHO_27-12-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2013/INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2013/INMET_N_PA_A234_TUCUMA_01-01-2013_A_31-12-2013.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2014/2014/INMET_N_PA_A234_TUCUMA_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2014/2014/INMET_NE_RN_A302_ARQ.SAO PEDRO E SAO PAULO_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2014/2014/INMET_S_RS_A804_SANTANA DO LIVRAMENTO_01-01-2014_A_31-12-2014.CSV: single positional indexer is out-of-bounds
Erro em dados-brutos/dados-climaticos/2022/INMET_N_TO_A049_COLINAS DO TOCANTINS_

In [39]:
df_temp.tail()

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude
6745,569.4,30.264793,18.981361,67.604282,206.2,387.00,CO,MS,BATAGUASSU,2016,"-21,75138888","-52,47055554"
6746,1140.8,28.080328,17.037978,62.260310,507.2,1159.54,CO,DF,BRASILIA,2016,"-15,78944444","-47,92583332"
6747,1618.8,26.074238,12.508310,77.403441,584.0,1150.00,SE,MG,CALDAS,2016,"-21,91805554","-46,38305554"
6748,1895.6,30.111202,17.782240,79.676700,620.0,591.00,CO,MT,COMODORO,2016,"-13,70861111","-59,76277777"
6749,617.0,32.302265,19.851133,69.279628,18.6,518.00,CO,MT,NOVA UBIRATA,2016,"-13,68638888","-54,95611111"


In [40]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # raio da Terra em km
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c 

In [41]:
mesorregioes

,mesorregiao,latitude,longitude
0,Madeira-Guaporé (RO),-10.302659,-64.055361
1,Leste Rondoniense (RO),-11.406309,-61.861723
2,Vale do Juruá (AC),-8.539777,-71.773867
3,Vale do Acre (AC),-9.941532,-69.066201
4,Norte Amazonense (AM),-0.615226,-65.567522
...,...,...,...
134,Norte Goiano (GO),-13.895544,-48.347095
135,Centro Goiano (GO),-15.981377,-49.778494
136,Leste Goiano (GO),-15.279044,-47.512262
137,Sul Goiano (GO),-17.743173,-50.504003


In [42]:
cols = [
        'latitude',
        'longitude'
    ]

for col in cols:
    df_temp[col] = pd.to_numeric(
        df_temp[col].astype(str).str.replace(',', '.'),
        errors='coerce'
    )

df_temp['latitude'] = df_temp['latitude'].astype(float)
df_temp['longitude'] = df_temp['longitude'].astype(float)

In [43]:
df_temp_teste = df_temp[df_temp['ano'] == 2019]

In [44]:
df_temp_teste

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude
4029,337.4,32.609497,22.196648,73.092404,334.4,147.75,CO,MT,PORTO ESTRELA,2019,-15.324722,-57.225833
4030,1038.2,32.326849,20.144658,69.628680,391.6,374.00,CO,MS,SELVIRIA,2019,-20.351444,-51.430222
4031,69.6,33.727072,20.518508,67.256851,1.6,515.30,NE,PI,CARACOL,2019,-9.285833,-43.324444
4032,1995.8,27.203836,16.233973,72.005500,580.0,245.50,S,RS,SAO LUIZ GONZAGA,2019,-28.417113,-54.962403
4033,479.8,32.939726,20.623562,73.671252,29.2,182.00,NE,BA,RIBEIRA DO AMPARO,2019,-11.058611,-38.444167
...,...,...,...,...,...,...,...,...,...,...,...,...
4612,1323.2,25.783014,15.574247,75.913470,395.0,106.99,S,RS,RIO PARDO,2019,-29.872113,-52.381980
4613,1036.6,31.443288,18.047945,64.427910,499.0,640.85,SE,MG,UNAI,2019,-16.554101,-46.881935
4614,1107.2,31.431621,24.712253,65.795276,19.6,3.72,NE,SE,ARACAJU,2019,-10.952413,-37.054330
4615,1252.8,28.580548,19.535616,79.014798,61.4,756.00,NE,CE,TIANGUA,2019,-3.732169,-41.011881


In [45]:
def achar_microrregiao(lat_est, lon_est, df_centroides):
    menor_distancia = float('inf')
    microrregiao_mais_proxima = None
    
    for _, row in df_centroides.iterrows():
        dist = haversine(lat_est, lon_est, row['latitude'], row['longitude'])
        
        if dist < menor_distancia:
            menor_distancia = dist
            microrregiao_mais_proxima = row['mesorregiao']
    
    return microrregiao_mais_proxima, menor_distancia


# Aplica para todas as estações
df_temp_teste['mesorregiao'] = df_temp_teste.apply(
    lambda row: achar_microrregiao(row['latitude'], row['longitude'], mesorregioes)[0],
    axis=1
)

print(mesorregioes)

                mesorregiao   latitude  longitude
0      Madeira-Guaporé (RO) -10.302659 -64.055361
1    Leste Rondoniense (RO) -11.406309 -61.861723
2        Vale do Juruá (AC)  -8.539777 -71.773867
3         Vale do Acre (AC)  -9.941532 -69.066201
4     Norte Amazonense (AM)  -0.615226 -65.567522
..                      ...        ...        ...
134       Norte Goiano (GO) -13.895544 -48.347095
135      Centro Goiano (GO) -15.981377 -49.778494
136       Leste Goiano (GO) -15.279044 -47.512262
137         Sul Goiano (GO) -17.743173 -50.504003
138   Distrito Federal (DF) -15.781166 -47.796851

[139 rows x 3 columns]


/var/folders/wp/hrfyx8x11jlbkyy3vr606yph0000gn/T/ipykernel_5045/3574635126.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp_teste['mesorregiao'] = df_temp_teste.apply(


In [46]:
df_temp_teste

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude,mesorregiao
4029,337.4,32.609497,22.196648,73.092404,334.4,147.75,CO,MT,PORTO ESTRELA,2019,-15.324722,-57.225833,Centro-Sul Mato-grossense (MT)
4030,1038.2,32.326849,20.144658,69.628680,391.6,374.00,CO,MS,SELVIRIA,2019,-20.351444,-51.430222,Araçatuba (SP)
4031,69.6,33.727072,20.518508,67.256851,1.6,515.30,NE,PI,CARACOL,2019,-9.285833,-43.324444,Sudoeste Piauiense (PI)
4032,1995.8,27.203836,16.233973,72.005500,580.0,245.50,S,RS,SAO LUIZ GONZAGA,2019,-28.417113,-54.962403,Centro Ocidental Rio-grandense (RS)
4033,479.8,32.939726,20.623562,73.671252,29.2,182.00,NE,BA,RIBEIRA DO AMPARO,2019,-11.058611,-38.444167,Nordeste Baiano (BA)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4612,1323.2,25.783014,15.574247,75.913470,395.0,106.99,S,RS,RIO PARDO,2019,-29.872113,-52.381980,Centro Oriental Rio-grandense (RS)
4613,1036.6,31.443288,18.047945,64.427910,499.0,640.85,SE,MG,UNAI,2019,-16.554101,-46.881935,Noroeste de Minas (MG)
4614,1107.2,31.431621,24.712253,65.795276,19.6,3.72,NE,SE,ARACAJU,2019,-10.952413,-37.054330,Leste Sergipano (SE)
4615,1252.8,28.580548,19.535616,79.014798,61.4,756.00,NE,CE,TIANGUA,2019,-3.732169,-41.011881,Noroeste Cearense (CE)


In [47]:
df_temp = df_temp.merge(
    df_temp_teste[['mesorregiao', 'regiao', 'uf', 'cidade']],
    on=['regiao', 'uf', 'cidade'],
    how='left'
)

In [48]:
df_temp

,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,regiao,uf,cidade,ano,latitude,longitude,mesorregiao
0,460.8,30.678470,19.179320,65.329483,87.4,680.70,NE,PE,ARCO VERDE,2013,-8.416944,-37.083333,Agreste Pernambucano (PE)
1,1557.2,22.654795,13.720274,78.815463,265.8,1106.00,S,PR,VENTANIA,2013,-24.238333,-50.245556,Centro Oriental Paranaense (PR)
2,1829.0,32.864932,22.069041,74.905142,470.2,203.00,N,PA,RONDON DO PARA,2013,-4.827500,-48.173333,Oeste Maranhense (MA)
3,799.4,33.267712,21.108464,67.730166,454.4,254.00,NE,MA,BALSAS,2013,-7.455278,-46.027500,Sul Maranhense (MA)
4,1366.8,31.558621,23.550345,84.601032,0.0,72.00,N,AM,HUMAITA,2013,-7.922778,-63.121389,Sul Amazonense (AM)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6745,569.4,30.264793,18.981361,67.604282,206.2,387.00,CO,MS,BATAGUASSU,2016,-21.751389,-52.470556,Presidente Prudente (SP)
6746,1140.8,28.080328,17.037978,62.260310,507.2,1159.54,CO,DF,BRASILIA,2016,-15.789444,-47.925833,Distrito Federal (DF)
6747,1618.8,26.074238,12.508310,77.403441,584.0,1150.00,SE,MG,CALDAS,2016,-21.918056,-46.383056,Sul/Sudoeste de Minas (MG)
6748,1895.6,30.111202,17.782240,79.676700,620.0,591.00,CO,MT,COMODORO,2016,-13.708611,-59.762778,Sudoeste Mato-grossense (MT)


In [49]:
df_temp['mesorregiao'].nunique()

139

In [50]:
mesorregioes['mesorregiao'].nunique()

139

In [51]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6750 entries, 0 to 6749
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   precipitacao_total (mm)  6750 non-null   float64
 1   temperatura_maxima (°C)  6750 non-null   float64
 2   temperatura_minima (°C)  6750 non-null   float64
 3   umidade_relativa (%)     6750 non-null   float64
 4   precipitacao_floracao    6750 non-null   float64
 5   altitude                 6750 non-null   float64
 6   regiao                   6750 non-null   object 
 7   uf                       6750 non-null   object 
 8   cidade                   6750 non-null   object 
 9   ano                      6750 non-null   int32  
 10  latitude                 6750 non-null   float64
 11  longitude                6750 non-null   float64
 12  mesorregiao              6607 non-null   object 
dtypes: float64(8), int32(1), object(4)
memory usage: 659.3+ KB


In [52]:
df_temp_agg = df_temp.groupby(['mesorregiao', 'ano']).agg({
    'precipitacao_total (mm)': 'mean',
    'temperatura_maxima (°C)': 'mean',
    'temperatura_minima (°C)': 'mean',
    'umidade_relativa (%)': 'mean',
    'precipitacao_floracao': 'mean',
    'altitude': 'mean'
}).reset_index()

In [53]:
df_temp_agg

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Agreste Alagoano (AL),2012,601.533333,30.401879,21.205324,73.892771,35.266667,196.666667
1,Agreste Alagoano (AL),2013,1007.266667,30.712952,21.571794,76.156624,224.200000,196.666667
2,Agreste Alagoano (AL),2014,1036.800000,30.174338,21.319909,78.302930,196.066667,196.666667
3,Agreste Alagoano (AL),2015,574.600000,32.988236,23.222919,71.140840,38.133333,196.666667
4,Agreste Alagoano (AL),2016,538.066667,31.911789,22.327992,65.495402,30.733333,196.666667
...,...,...,...,...,...,...,...,...
1769,Zona da Mata (MG),2020,1660.733333,29.007442,18.022448,78.142019,505.666667,463.856667
1770,Zona da Mata (MG),2021,896.600000,29.200172,17.050388,77.010000,349.800000,463.856667
1771,Zona da Mata (MG),2022,1326.733333,29.056661,17.429160,76.393595,734.600000,463.856667
1772,Zona da Mata (MG),2023,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667


In [54]:
df_temp_agg[df_temp_agg['mesorregiao'].str.contains('(SC)')]

/var/folders/wp/hrfyx8x11jlbkyy3vr606yph0000gn/T/ipykernel_5045/3514436738.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_temp_agg[df_temp_agg['mesorregiao'].str.contains('(SC)')]


,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
433,Grande Florianópolis (SC),2012,1393.00,25.562842,18.126503,77.570397,284.80,1.800
434,Grande Florianópolis (SC),2013,1601.80,24.890137,17.440822,77.828063,262.80,1.800
435,Grande Florianópolis (SC),2014,1530.20,25.803836,18.530685,78.482802,388.60,1.800
436,Grande Florianópolis (SC),2015,2275.40,25.530959,18.756438,80.173554,692.40,1.800
437,Grande Florianópolis (SC),2016,1390.00,22.229930,13.749866,80.844010,585.90,441.400
...,...,...,...,...,...,...,...,...
1705,Vale do Itajaí (SC),2020,1034.40,24.668980,15.302514,82.191869,292.95,288.365
1706,Vale do Itajaí (SC),2021,760.05,23.235632,15.737916,85.148857,76.35,288.365
1707,Vale do Itajaí (SC),2022,1171.70,26.073810,16.485651,80.541036,577.70,288.365
1708,Vale do Itajaí (SC),2023,1843.35,25.063638,16.165525,84.111455,468.25,288.365


In [55]:
df_final.head()

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0,36000.0,2.0,2.0,2000.0,-7.029713,-35.819545
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0,3851000.0,2733.0,2733.0,1075000.0,-8.510505,-36.416511
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0,2439000.0,2615.0,2615.0,771000.0,-8.510505,-36.416511
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1618000.0,2549.0,2549.0,693000.0,-8.510505,-36.416511
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0,2325000.0,2265.0,2265.0,867000.0,-8.510505,-36.416511


In [56]:
df_temp_agg.head()

,mesorregiao,ano,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Agreste Alagoano (AL),2012,601.533333,30.401879,21.205324,73.892771,35.266667,196.666667
1,Agreste Alagoano (AL),2013,1007.266667,30.712952,21.571794,76.156624,224.200000,196.666667
2,Agreste Alagoano (AL),2014,1036.800000,30.174338,21.319909,78.302930,196.066667,196.666667
3,Agreste Alagoano (AL),2015,574.600000,32.988236,23.222919,71.140840,38.133333,196.666667
4,Agreste Alagoano (AL),2016,538.066667,31.911789,22.327992,65.495402,30.733333,196.666667


In [57]:
df_final['ano'] = df_final['ano'].astype(int)

df_final = df_final.merge(df_temp_agg, on=['mesorregiao', 'ano'], how='left')

In [58]:
df_final

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0,36000.0,2.0,2.0,2000.0,-7.029713,-35.819545,768.800000,28.629075,20.254114,82.511918,54.100000,559.810000
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0,3851000.0,2733.0,2733.0,1075000.0,-8.510505,-36.416511,232.066667,29.143534,18.353461,70.905596,12.333333,684.486667
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0,2439000.0,2615.0,2615.0,771000.0,-8.510505,-36.416511,618.066667,29.036885,18.879336,73.331692,106.800000,684.486667
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1618000.0,2549.0,2549.0,693000.0,-8.510505,-36.416511,606.266667,28.628146,18.599447,74.939452,116.933333,684.486667
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0,2325000.0,2265.0,2265.0,867000.0,-8.510505,-36.416511,403.000000,29.760529,18.845805,70.811290,57.933333,684.486667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224,Zona da Mata (MG),2020,Café (em grão) Canephora,2362.0,6623000.0,522.0,522.0,1233000.0,-21.010901,-42.819740,1660.733333,29.007442,18.022448,78.142019,505.666667,463.856667
1225,Zona da Mata (MG),2021,Café (em grão) Canephora,2121.0,8679000.0,504.0,504.0,1069000.0,-21.010901,-42.819740,896.600000,29.200172,17.050388,77.010000,349.800000,463.856667
1226,Zona da Mata (MG),2022,Café (em grão) Canephora,2746.0,15327000.0,488.0,488.0,1340000.0,-21.010901,-42.819740,1326.733333,29.056661,17.429160,76.393595,734.600000,463.856667
1227,Zona da Mata (MG),2023,Café (em grão) Canephora,2371.0,10643000.0,498.0,498.0,1181000.0,-21.010901,-42.819740,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667


In [59]:
df_final[df_final['altitude'].isna()]

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
51,Assis (SP),2021,Café (em grão) Arábica,1608.0,2.917180e+08,17432.0,17415.0,28009000.0,-22.768414,-50.121347,NaN,NaN,NaN,NaN,NaN,NaN
52,Assis (SP),2022,Café (em grão) Arábica,1579.0,3.054490e+08,17437.0,17420.0,27514000.0,-22.768414,-50.121347,NaN,NaN,NaN,NaN,NaN,NaN
77,Campinas (SP),2021,Café (em grão) Arábica,1589.0,1.254622e+09,53648.0,53532.0,85049000.0,-22.220282,-46.989825,NaN,NaN,NaN,NaN,NaN,NaN
218,Centro-Sul Paranaense (PR),2012,Café (em grão) Arábica,1000.0,5.000000e+03,1.0,1.0,1000.0,-25.487813,-51.972895,NaN,NaN,NaN,NaN,NaN,NaN
477,Norte Fluminense (RJ),2012,Café (em grão) Arábica,1438.0,4.400000e+05,105.0,105.0,151000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
478,Norte Fluminense (RJ),2013,Café (em grão) Arábica,1180.0,3.430000e+05,89.0,89.0,105000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
479,Norte Fluminense (RJ),2014,Café (em grão) Arábica,1540.0,2.740000e+05,93.0,63.0,97000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
480,Norte Fluminense (RJ),2015,Café (em grão) Arábica,1540.0,2.740000e+05,63.0,63.0,97000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
481,Norte Fluminense (RJ),2016,Café (em grão) Arábica,1540.0,4.090000e+05,63.0,63.0,97000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN
482,Norte Fluminense (RJ),2017,Café (em grão) Arábica,1585.0,7.200000e+05,65.0,65.0,103000.0,-21.813031,-41.501991,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
df_final = df_final.sort_values(by=['mesorregiao', 'ano', 'tipo']).reset_index(drop=True)

df_final['precipitacao_total (mm)'] = df_final['precipitacao_total (mm)'].bfill()
df_final['temperatura_maxima (°C)'] = df_final['temperatura_maxima (°C)'].bfill()
df_final['temperatura_minima (°C)'] = df_final['temperatura_minima (°C)'].bfill()
df_final['umidade_relativa (%)'] = df_final['umidade_relativa (%)'].bfill()
df_final['altitude'] = df_final['altitude'].bfill()

In [61]:
df_final[df_final['altitude'].isna()]

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude


In [62]:
df_final[df_final['mesorregiao'].str.contains('Sudoeste Amazonense')]

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude
904,Sudoeste Amazonense (AM),2012,Café (em grão) Canephora,1000.0,241000.0,131.0,108.0,108000.0,-5.180785,-69.35668,693.200000,32.257724,22.260976,81.921297,608.8,143.00
905,Sudoeste Amazonense (AM),2013,Café (em grão) Canephora,800.0,168000.0,108.0,70.0,56000.0,-5.180785,-69.35668,2369.000000,31.049315,22.024384,80.183440,814.0,143.00
906,Sudoeste Amazonense (AM),2014,Café (em grão) Canephora,1200.0,266000.0,80.0,80.0,96000.0,-5.180785,-69.35668,2218.600000,30.990909,22.102204,70.406612,275.2,143.00
907,Sudoeste Amazonense (AM),2015,Café (em grão) Canephora,1200.0,137000.0,30.0,30.0,36000.0,-5.180785,-69.35668,2779.400000,31.900548,22.301644,65.677770,808.0,143.00
908,Sudoeste Amazonense (AM),2016,Café (em grão) Canephora,1200.0,137000.0,30.0,30.0,36000.0,-5.180785,-69.35668,786.400000,31.425743,23.499010,71.101247,0.0,143.00
909,Sudoeste Amazonense (AM),2017,Café (em grão) Canephora,1200.0,162000.0,30.0,30.0,36000.0,-5.180785,-69.35668,96.400000,31.576667,23.525000,72.051523,NaN,121.54
910,Sudoeste Amazonense (AM),2020,Café (em grão) Canephora,1194.0,144000.0,31.0,31.0,37000.0,-5.180785,-69.35668,96.400000,31.576667,23.525000,72.051523,0.0,121.54
911,Sudoeste Amazonense (AM),2021,Café (em grão) Arábica,1300.0,128000.0,21.0,10.0,13000.0,-5.180785,-69.35668,1171.733333,31.669913,19.965564,73.463972,NaN,377.25
912,Sudoeste Amazonense (AM),2022,Café (em grão) Arábica,2000.0,67000.0,4.0,4.0,8000.0,-5.180785,-69.35668,1171.733333,31.669913,19.965564,73.463972,NaN,377.25
913,Sudoeste Amazonense (AM),2022,Café (em grão) Canephora,1250.0,79000.0,19.0,12.0,15000.0,-5.180785,-69.35668,1171.733333,31.669913,19.965564,73.463972,NaN,377.25


In [63]:
df_final['aproveitamento_colheita'] = df_final['Variável - Área colhida (Hectares)'] / df_final['Variável - Área destinada à colheita (Hectares)']

df_final['amplitude_termica'] = df_final['temperatura_maxima (°C)'] - df_final['temperatura_minima (°C)']

df_final['preco_medio_kg'] = df_final['Variável - Valor da produção (Reais)'] / df_final['Variável - Quantidade produzida (Quilogramas)']

In [64]:
df_final

,mesorregiao,ano,tipo,Variável - Rendimento médio da produção (Quilogramas por Hectare),Variável - Valor da produção (Reais),Variável - Área destinada à colheita (Hectares),Variável - Área colhida (Hectares),Variável - Quantidade produzida (Quilogramas),latitude,longitude,precipitacao_total (mm),temperatura_maxima (°C),temperatura_minima (°C),umidade_relativa (%),precipitacao_floracao,altitude,aproveitamento_colheita,amplitude_termica,preco_medio_kg
0,Agreste Paraibano (PB),2024,Café (em grão) Arábica,1000.0,3.600000e+04,2.0,2.0,2000.0,-7.029713,-35.819545,768.800000,28.629075,20.254114,82.511918,54.100000,559.810000,1.0,8.374962,18.000000
1,Agreste Pernambucano (PE),2012,Café (em grão) Arábica,393.0,3.851000e+06,2733.0,2733.0,1075000.0,-8.510505,-36.416511,232.066667,29.143534,18.353461,70.905596,12.333333,684.486667,1.0,10.790073,3.582326
2,Agreste Pernambucano (PE),2013,Café (em grão) Arábica,295.0,2.439000e+06,2615.0,2615.0,771000.0,-8.510505,-36.416511,618.066667,29.036885,18.879336,73.331692,106.800000,684.486667,1.0,10.157548,3.163424
3,Agreste Pernambucano (PE),2014,Café (em grão) Arábica,272.0,1.618000e+06,2549.0,2549.0,693000.0,-8.510505,-36.416511,606.266667,28.628146,18.599447,74.939452,116.933333,684.486667,1.0,10.028699,2.334776
4,Agreste Pernambucano (PE),2015,Café (em grão) Arábica,383.0,2.325000e+06,2265.0,2265.0,867000.0,-8.510505,-36.416511,403.000000,29.760529,18.845805,70.811290,57.933333,684.486667,1.0,10.914724,2.681661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224,Zona da Mata (MG),2022,Café (em grão) Canephora,2746.0,1.532700e+07,488.0,488.0,1340000.0,-21.010901,-42.819740,1326.733333,29.056661,17.429160,76.393595,734.600000,463.856667,1.0,11.627501,11.438060
1225,Zona da Mata (MG),2023,Café (em grão) Arábica,1299.0,3.658570e+09,201832.0,201832.0,262264000.0,-21.010901,-42.819740,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667,1.0,11.376530,13.949951
1226,Zona da Mata (MG),2023,Café (em grão) Canephora,2371.0,1.064300e+07,498.0,498.0,1181000.0,-21.010901,-42.819740,1327.266667,29.921461,18.544932,77.370603,405.400000,463.856667,1.0,11.376530,9.011854
1227,Zona da Mata (MG),2024,Café (em grão) Arábica,1376.0,5.693510e+09,205375.0,205375.0,282595000.0,-21.010901,-42.819740,1511.800000,30.457941,18.854546,76.623899,654.066667,463.856667,1.0,11.603395,20.147243
